# IRC Vendaval — Resultados por Técnica

Apresentação visual de cada técnica testada para correção de viés de rajadas (ERA5 → INMET).

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from IPython.display import display

ROOT = Path("..")  # raiz do projeto (notebook está em notebooks/)
LAZY = ROOT / "artifacts" / "lazy_modal" / "lazy_clusters"
GAN  = ROOT / "artifacts" / "gan_modal"  / "gan_clusters"

CLUSTERS = [1, 2, 3, 4, 5, 6]

def show_images(paths, titles=None, cols=3, figsize_per=(5, 4)):
    """Exibe lista de imagens em grade."""
    paths = [Path(p) for p in paths]
    paths = [p for p in paths if p.exists()]
    if not paths:
        print("  (nenhuma imagem encontrada)")
        return
    rows = (len(paths) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols,
                             figsize=(figsize_per[0]*cols, figsize_per[1]*rows))
    axes = [axes] if rows == 1 and cols == 1 else axes
    axes = [ax for row in (axes if rows > 1 else [axes]) for ax in (row if cols > 1 else [row])]
    for i, (ax, p) in enumerate(zip(axes, paths)):
        ax.imshow(mpimg.imread(p))
        ax.axis("off")
        ax.set_title(titles[i] if titles else p.stem, fontsize=9)
    for ax in axes[len(paths):]:
        ax.set_visible(False)
    plt.tight_layout()
    plt.show()

print("Setup OK")

---
## 1. Dados — Distribuição do Target por Cluster

Visualização da distribuição da rajada máxima observada (`daily_wind_gust_max`) e da
diferença sistemática em relação ao proxy ERA5 (`wind_mag_max`) em cada cluster.

In [ ]:
import sys
sys.path.insert(0, str(ROOT))

from src.data.netcdf_loader import NetCDFLoader
from src.data.cluster_assigner import assign_station_clusters
from src.pipelines.common import build_flat_dataframe, make_split, TRAIN_SLICE
from src.data.climatology import build_era5_climatology

loader = NetCDFLoader(ROOT / "dataset" / "raw")
ds_inmet, ds_era5 = loader.load()
station_clusters = assign_station_clusters(ds_inmet, ROOT / "dataset" / "shp")
ds_clim = build_era5_climatology(ds_era5, station_clusters)
df = build_flat_dataframe(ds_inmet, ds_era5, station_clusters, ds_clim)
df_tr = make_split(df, TRAIN_SLICE)
print(f"Total de amostras (treino): {len(df_tr):,}")
print(f"Clusters: {sorted(df_tr['cluster_id'].unique())}")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, cid in zip(axes.flat, CLUSTERS):
    sub = df_tr[df_tr["cluster_id"] == cid]
    ax.hist(sub["daily_wind_gust_max"], bins=40, alpha=0.7,
            color="steelblue", label="INMET (alvo)")
    ax.hist(sub["wind_mag_max"], bins=40, alpha=0.5,
            color="tomato", label="ERA5 (proxy)")
    ax.axvline(sub["daily_wind_gust_max"].quantile(0.9),
               color="navy", ls="--", lw=1, label="P90 INMET")
    ax.set_title(f"Cluster {cid}  (n={len(sub):,})")
    ax.set_xlabel("m/s")
    ax.legend(fontsize=7)
fig.suptitle("Distribuição INMET vs ERA5 — treino 2000–2022", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Viés ERA5 por cluster
bias = (df_tr.groupby("cluster_id")
        .apply(lambda g: (g["wind_mag_max"] - g["daily_wind_gust_max"]).mean())
        .rename("bias_era5"))

fig, ax = plt.subplots(figsize=(7, 4))
colors = ["tomato" if b < 0 else "steelblue" for b in bias]
ax.bar([f"C{c}" for c in bias.index], bias.values, color=colors, edgecolor="k", lw=0.5)
ax.axhline(0, color="k", lw=1)
ax.set_ylabel("ERA5 − INMET (m/s)")
ax.set_title("Viés sistemático ERA5 por cluster (treino)\n"
             "Negativo = ERA5 subestima a rajada observada")
plt.tight_layout()
plt.show()

---
## 2. GAN Augmentation — cWGAN-GP por Cluster

O modelo gerador (`convergence_test15`) foi treinado para gerar rajadas sintéticas
condicionadas ao cluster. Apenas amostras acima do P90 são retidas (rejection sampling).

In [ ]:
# Curva de loss da GAN (convergence_test14 tem loss_curve salvo)
loss_curve = GAN / "convergence_test14" / "plots" / "loss_curve.png"
show_images([loss_curve], ["Wasserstein Loss — GAN treino"], cols=1, figsize_per=(9, 4))

In [ ]:
# Distribuição real vs sintético por cluster (convergence_test15)
gan_dist_plots = [GAN / "convergence_test15" / "plots" / f"cluster_{c}.png"
                  for c in CLUSTERS]
show_images(gan_dist_plots, [f"Cluster {c}" for c in CLUSTERS], cols=3, figsize_per=(5, 3.5))

In [ ]:
# Volume de sintéticos gerados por cluster
synth = pd.read_csv(GAN / "convergence_test15" / "synthetic_augment.csv")
print(f"Total de amostras sintéticas: {len(synth):,}")
display(synth.groupby("cluster_id").size().rename("n_synthetic").to_frame())

---
## 3. LazyPredict — Screening de 30+ Modelos por Cluster

Avaliação simultânea de todos os regressores sklearn. Experimento `feat_eng_v1`
(26 features, com augmentation GAN).

In [ ]:
# Heatmap: R² de todos os modelos × clusters
show_images([LAZY / "feat_eng_v1" / "plots" / "r2_heatmap.png"],
            ["R² — todos os modelos × clusters (feat_eng_v1)"],
            cols=1, figsize_per=(12, 6))

In [ ]:
# Top-5 modelos por cluster
top5_paths = [LAZY / "feat_eng_v1" / "plots" / f"top5_c{c}.png" for c in CLUSTERS]
show_images(top5_paths, [f"Top-5 — Cluster {c}" for c in CLUSTERS], cols=3, figsize_per=(5, 3.5))

In [ ]:
# Scatter melhor modelo por cluster (validação)
scatter_paths = [LAZY / "feat_eng_v1" / "plots" / f"scatter_cc{c}.png" for c in CLUSTERS]
show_images(scatter_paths, [f"Scatter val — C{c}" for c in CLUSTERS], cols=3, figsize_per=(5, 4))

---
## 4. Evolução do R² entre Técnicas

Comparação do R² do melhor modelo por cluster ao longo das iterações do projeto.

In [ ]:
def best_r2_per_cluster(exp_name):
    csv = LAZY / exp_name / "lazy_cluster_results.csv"
    if not csv.exists():
        return None
    df = pd.read_csv(csv)
    col = "R-Squared" if "R-Squared" in df.columns else "R2_deploy_mean"
    return df.groupby("cluster_id")[col].max()

experiments = {
    "gan_synth\n(baseline)": "gan_synth",
    "feat_eng_v1\n(26 features)": "feat_eng_v1",
    "neighbor_v2\n(+vizinhos)": "neighbor_v2",
    "deploy_eval_v1\n(+deploy eval)": "deploy_eval_v1",
    "temporal_v1\n(+lags obs)": "temporal_v1",
}

records = []
for label, exp in experiments.items():
    r2 = best_r2_per_cluster(exp)
    if r2 is not None:
        for cid, val in r2.items():
            records.append({"Experimento": label, "Cluster": f"C{cid}", "R²": val})

evo = pd.DataFrame(records)

fig, ax = plt.subplots(figsize=(13, 5))
pivot = evo.pivot(index="Experimento", columns="Cluster", values="R²")
pivot.plot(kind="bar", ax=ax, width=0.75, colormap="tab10")
ax.set_ylabel("R² (melhor modelo por cluster)")
ax.set_title("Evolução do R² entre técnicas")
ax.axhline(0.7, color="k", ls="--", lw=1, label="Meta R²=0.70")
ax.legend(loc="lower right", fontsize=8)
ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha="right")
plt.tight_layout()
plt.show()

display(pivot.round(3))

---
## 5. Impacto da Augmentation GAN

Diferença de R² com e sem dados sintéticos para cada modelo (verde = melhora).

In [ ]:
synth_comp = [LAZY / "temporal_v1" / "plots" / f"synth_comparison_c{c}.png"
              for c in CLUSTERS]
show_images(synth_comp, [f"ΔR² (com−sem GAN) — C{c}" for c in CLUSTERS],
            cols=3, figsize_per=(5, 3.5))

In [ ]:
# Distribuição treino: real vs sintético
dist_paths = [LAZY / "temporal_v1" / "plots" / f"train_distribution_c{c}.png"
              for c in CLUSTERS]
show_images(dist_paths, [f"Distribuição treino — C{c}" for c in CLUSTERS],
            cols=3, figsize_per=(5, 3.5))

---
## 6. Avaliação por Janela de Deploy (Mensal)

R² calculado mês a mês sobre o ano de validação (2023). Revela instabilidade
sazonal que o R² anual mascara.

In [ ]:
rolling_paths = [LAZY / "deploy_eval_v1" / "plots" / f"rolling_r2_c{c}.png"
                 for c in CLUSTERS]
show_images(rolling_paths, [f"R² mensal — C{c}" for c in CLUSTERS],
            cols=3, figsize_per=(5, 3.5))

In [ ]:
# Resumo estatístico do R² de deploy por cluster
deploy_df = pd.read_csv(LAZY / "deploy_eval_v1" / "lazy_cluster_results.csv")
best_deploy = (deploy_df.sort_values("R2_deploy_mean", ascending=False)
               .groupby("cluster_id")
               .first()[["Model", "R-Squared", "R2_deploy_mean",
                         "R2_deploy_std", "R2_deploy_min"]])
display(best_deploy.round(3))

---
## 7. Compartilhamento entre Clusters Vizinhos

Dados reais e sintéticos do cluster vizinho mais próximo (distância euclidiana
entre centroides) são incluídos no treino. Experimento `neighbor_v2`.

In [ ]:
show_images([LAZY / "neighbor_v2" / "plots" / "r2_heatmap.png"],
            ["Heatmap R² — neighbor_v2 (1 vizinho)"],
            cols=1, figsize_per=(12, 6))

In [ ]:
# Scatter melhor modelo por cluster
neighbor_scatters = [LAZY / "neighbor_v2" / "plots" / f"scatter_cc{c}.png"
                     for c in CLUSTERS]
show_images(neighbor_scatters, [f"Scatter val — C{c}" for c in CLUSTERS],
            cols=3, figsize_per=(5, 4))

---
## 8. Estratificação por Trimestre Climático

Um modelo independente por cluster × trimestre (DJF/MAM/JJA/SON).
Scatter 2×2 do melhor modelo em `temporal_v1`.

In [ ]:
seasonal_scatters = [LAZY / "temporal_v1" / "plots" / f"seasonal_scatter_c{c}.png"
                     for c in CLUSTERS]
show_images(seasonal_scatters, [f"Scatter sazonal — C{c}" for c in CLUSTERS],
            cols=2, figsize_per=(7, 6))

---
## 9. Conectividade Temporal (Features AR da Observação)

Lags da observação INMET (`lag1/2/3_gust_obs`), rolling 7d, anomalia ERA5
e lag de tendência de pressão. Experimento `temporal_v1`.

In [ ]:
# R² de deploy mensal com features temporais
temporal_rolling = [LAZY / "temporal_v1" / "plots" / f"rolling_r2_c{c}.png"
                    for c in CLUSTERS]
show_images(temporal_rolling, [f"R² mensal (temporal) — C{c}" for c in CLUSTERS],
            cols=3, figsize_per=(5, 3.5))

In [ ]:
# Top-5 com features temporais
top5_temporal = [LAZY / "temporal_v1" / "plots" / f"top5_c{c}.png" for c in CLUSTERS]
show_images(top5_temporal, [f"Top-5 (temporal) — C{c}" for c in CLUSTERS],
            cols=3, figsize_per=(5, 3.5))

---
## 10. Resumo Final

Comparação do melhor R² por cluster entre todos os experimentos com resultados disponíveis.

In [ ]:
if not evo.empty:
    fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharey=True)
    for ax, cid in zip(axes.flat, CLUSTERS):
        sub = evo[evo["Cluster"] == f"C{cid}"]
        if sub.empty:
            ax.set_visible(False)
            continue
        bars = ax.barh(sub["Experimento"], sub["R²"], color="steelblue", edgecolor="k", lw=0.4)
        ax.axvline(0.7, color="tomato", ls="--", lw=1)
        ax.set_xlim(0, 1)
        ax.set_title(f"Cluster {cid}", fontweight="bold")
        ax.set_xlabel("R²")
        for bar, val in zip(bars, sub["R²"]):
            ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
                    f"{val:.3f}", va="center", fontsize=7)
    fig.suptitle("Evolução R² — melhor modelo por cluster e técnica", fontsize=13)
    plt.tight_layout()
    plt.show()

---
## 11. Mapas Espaciais de Correção

Adaptação do estilo visual do projeto paralelo (`plot.py`): dark theme, painéis
side-by-side, colorbars individuais. As predições usam o **melhor modelo LazyPredict
por cluster** (máximo R² na validação 2023).

In [ ]:
import sys
sys.path.insert(0, str(ROOT))
import geopandas as gpd
from src.pipelines.common import TEST_SLICE
from src.visualization.spatial_plots import (
    build_spatial_df, plot_correction_panels,
    plot_difference_map, plot_r2_choropleth, plot_extreme_map,
)

# Shapefile dos clusters (reprojetado para WGS-84)
gdf = gpd.read_file(ROOT / 'dataset' / 'shp' / 'shp_vento.shp').to_crs(4326)
gdf_active = gdf[gdf['cluster'].astype(int) <= 6].copy()

# Flat DataFrame do split de validação (lat/lon disponível)
df_val = make_split(df, TEST_SLICE)  # df já construído na seção 1

# Experimento com todas as técnicas acumuladas
EXP = LAZY / 'temporal_v1'

# Montar DataFrame espacial: predição do melhor modelo + lat/lon
df_sp = build_spatial_df(df_val, EXP)
print(f'Amostras espaciais: {len(df_sp):,}  |  Estações: {df_sp["estacao"].nunique()}')
print('Melhor modelo por cluster:')
print(df_sp.groupby('cluster_id')['best_model'].first().to_string())

OUT = ROOT / 'artifacts' / 'plots' / 'spatial'
OUT.mkdir(parents=True, exist_ok=True)

### 11.1 ERA5 | Corrigido | INMET — Média por Estação

Três painéis com scatter das estações colorido pela rajada média (2023).
Escala de cores compartilhada para comparação direta.

In [ ]:
plot_correction_panels(df_sp, gdf_active, OUT / 'correction_mean.png', stat='mean')
from IPython.display import Image
Image(OUT / 'correction_mean.png', width=900)

### 11.2 ERA5 | Corrigido | INMET — P90 por Estação

Mesmo layout, mas usando o percentil 90 por estação — foco nos eventos extremos.

In [ ]:
plot_extreme_map(df_sp, gdf_active, OUT / 'extreme_p90.png', quantile=0.90)
Image(OUT / 'extreme_p90.png', width=900)

### 11.3 Diagnóstico: Correção ML vs Viés Real

Dois painéis divergentes (azul = subestima, vermelho = superestima):
- **Correção aplicada** = y_pred − ERA5
- **Viés real** = INMET − ERA5

Se os padrões espaciais coincidirem, o modelo está corrigindo na direção certa.

In [ ]:
plot_difference_map(df_sp, gdf_active, OUT / 'difference_map.png')
Image(OUT / 'difference_map.png', width=700)

### 11.4 Mapa de R² por Cluster

Polígonos dos 6 clusters coloridos pelo R² do melhor modelo (verde = melhor).
Estações como pontos brancos sobrepostos.

In [ ]:
results_df = pd.read_csv(EXP / 'lazy_cluster_results.csv')
plot_r2_choropleth(results_df, gdf_active, OUT / 'r2_choropleth.png', df_sp=df_sp)
Image(OUT / 'r2_choropleth.png', width=600)